# Markdown 入库练习

这个 Notebook 用于逐步观察、设计和验证 RAG 的文档分片。现在只包含**第 1 步：观察原文结构**；后续代码会在我们讨论并确认分片规则后补充。

请从上到下逐个运行单元。教材原文位于被 Git 忽略的 `data/raw/`；提交 Notebook 前请清除含教材正文的输出。

In [ ]:
from pathlib import Path

current = Path.cwd().resolve()
repo_root = next((path for path in (current, *current.parents) if (path / 'config' / 'sources.yaml').is_file()), None)
if repo_root is None:
    raise FileNotFoundError('请从项目目录或其子目录启动 Notebook')

sources = {
    'zh': repo_root / 'data/raw/d2l-zh/chapter_attention-mechanisms/transformer.md',
    'en': repo_root / 'data/raw/d2l-en/chapter_attention-mechanisms-and-transformers/transformer.md',
}
print('项目目录:', repo_root)
for language, path in sources.items():
    print(f'{language}: 存在={path.is_file()}, 路径={path.relative_to(repo_root)}')
    if not path.is_file():
        raise FileNotFoundError(f'缺少本地教材文件: {path}')

## 第 1 步：观察原文

先查看中文文件的开头，注意标题、`:label:` 锚点，以及可能出现的 D2L 执行代码块。把 `language` 改成 `'en'` 可以查看英文文件。

In [ ]:
language = 'zh'
source_path = sources[language]
lines = source_path.read_text(encoding='utf-8').splitlines()
print(f'{language}: {len(lines)} 行；前 20 行：')
for number, line in enumerate(lines[:20], start=1):
    print(f'{number:>4}: {line}')

In [ ]:
import re

heading_pattern = re.compile(r'^(#{1,6})[ \t]+(.+)')
structure = []
for number, line in enumerate(lines, start=1):
    heading = heading_pattern.match(line)
    if heading:
        structure.append((number, f'H{len(heading.group(1))}', heading.group(2)))
    elif line.startswith(':label:'):
        structure.append((number, '锚点', line))

print(f'找到 {len(structure)} 个标题或锚点；前 30 个：')
for number, kind, value in structure[:30]:
    print(f'{number:>4}  {kind:<4} {value}')

## 下一步

把上面结构列表中的前几个标题、锚点和你观察到的特殊格式发给我。我们再一起确定分片边界，并为分片函数写第一个失败测试。